<a href="https://colab.research.google.com/github/pranav00167/Cloud-computing/blob/main/lab_assignment_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Assignment 6 -Pranav Bhardwaj 8025320073

**PCS221 Cloud Computing - Map Reduce (Without EMR)**

This notebook implements each problem using:

1. Manual Python MapReduce approach (`Mapper -> Shuffle -> Reducer`)
2. `mrjob` framework

For Q5 and Q9, place the required dataset files locally and update the file paths before running those cells.

In [ ]:
!pip install mrjob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 1.9 MB/s eta 0:00:00


In [ ]:
def shuffle_pairs(mapped_pairs):
    grouped = defaultdict(list)
    for key, value in mapped_pairs:
        grouped[key].append(value)
    return grouped

def run_manual_mapreduce(data, mapper, reducer):
    mapped = []
    for item in data:
        mapped.extend(mapper(item))
    grouped = shuffle_pairs(mapped)
    reduced = {}
    for key, values in grouped.items():
        reduced[key] = reducer(key, values)
    return reduced

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def print_result(title, result):
    print(f"\n{title}")
    if isinstance(result, dict):
        for key, value in sorted(result.items()):
            print(f"{key}: {value}")
    else:
        print(result)

## Q1. Word Count
Input:
- `hadoop is fast`
- `hadoop is scalable`

In [ ]:
q1_lines = ["hadoop is fast", "hadoop is scalable"]

def q1_mapper(line):
    return [(word, 1) for word in tokenize(line)]

def q1_reducer(word, counts):
    return sum(counts)

q1_manual = run_manual_mapreduce(q1_lines, q1_mapper, q1_reducer)
print_result("Q1 Manual Output", q1_manual)

if MRJOB_AVAILABLE:
    class MRWordCount(MRJob):
        def mapper(self, _, line):
            for word in tokenize(line):
                yield word, 1

        def reducer(self, word, counts):
            yield word, sum(counts)

    print("\nQ1 mrjob class ready: MRWordCount")


Q1 Manual Output
fast: 1
hadoop: 2
is: 2
scalable: 1

Q1 mrjob class ready: MRWordCount


## Q2. Character Count (ignore spaces)
Input: `big data`

In [ ]:
q2_text = "big data"

def q2_mapper(text):
    return [(char.lower(), 1) for char in text if not char.isspace()]

def q2_reducer(char, counts):
    return sum(counts)

q2_manual = run_manual_mapreduce([q2_text], q2_mapper, q2_reducer)
print_result("Q2 Manual Output", q2_manual)

if MRJOB_AVAILABLE:
    class MRCharacterCount(MRJob):
        def mapper(self, _, line):
            for char in line:
                if not char.isspace():
                    yield char.lower(), 1

        def reducer(self, char, counts):
            yield char, sum(counts)

    print("\nQ2 mrjob class ready: MRCharacterCount")


Q2 Manual Output
a: 2
b: 1
d: 1
g: 1
i: 1
t: 1

Q2 mrjob class ready: MRCharacterCount


## Q3. Average Word Length (Per Word)
Input: `data science data big data`

In [ ]:
q3_text = "data science data big data"

def q3_mapper(text):
    return [(word, len(word)) for word in tokenize(text)]

def q3_reducer(word, lengths):
    return sum(lengths) / len(lengths)

q3_manual = run_manual_mapreduce([q3_text], q3_mapper, q3_reducer)
print_result("Q3 Manual Output", q3_manual)

if MRJOB_AVAILABLE:
    class MRAverageWordLengthPerWord(MRJob):
        def mapper(self, _, line):
            for word in tokenize(line):
                yield word, len(word)

        def reducer(self, word, lengths):
            lengths = list(lengths)
            yield word, sum(lengths) / len(lengths)

    print("\nQ3 mrjob class ready: MRAverageWordLengthPerWord")


Q3 Manual Output
big: 3.0
data: 4.0
science: 7.0

Q3 mrjob class ready: MRAverageWordLengthPerWord


## Q4. Global Average Word Length
Input: `hadoop mapreduce spark`

In [ ]:
q4_text = "hadoop mapreduce spark"

def q4_mapper(text):
    return [("global_avg", (len(word), 1)) for word in tokenize(text)]

def q4_reducer(_, values):
    total_length = sum(length for length, _ in values)
    total_words = sum(count for _, count in values)
    return total_length / total_words

q4_manual = run_manual_mapreduce([q4_text], q4_mapper, q4_reducer)
print_result("Q4 Manual Output", q4_manual)

if MRJOB_AVAILABLE:
    class MRGlobalAverageWordLength(MRJob):
        def mapper(self, _, line):
            for word in tokenize(line):
                yield "global_avg", (len(word), 1)

        def reducer(self, key, values):
            total_length = 0
            total_words = 0
            for length, count in values:
                total_length += length
                total_words += count
            yield key, total_length / total_words

    print("\nQ4 mrjob class ready: MRGlobalAverageWordLength")


Q4 Manual Output
global_avg: 6.666666666666667

Q4 mrjob class ready: MRGlobalAverageWordLength


## Q5. Perform Q1-Q4 on a file and find Top 5 most frequent words

Update `q5_file_path` to your local text file path downloaded from the Google Drive link in the assignment.

In [ ]:
q5_file_path = Path("sample_q5_input.txt")

def load_words_from_text_file(path):
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    return tokenize(text)

if q5_file_path.exists():
    q5_words = load_words_from_text_file(q5_file_path)
    q5_word_counts = Counter(q5_words)
    q5_char_counts = Counter(char for char in ''.join(q5_words))
    q5_per_word_avg = defaultdict(list)
    for word in q5_words:
        q5_per_word_avg[word].append(len(word))
    q5_per_word_avg = {word: sum(vals) / len(vals) for word, vals in q5_per_word_avg.items()}
    q5_global_avg = sum(len(word) for word in q5_words) / len(q5_words) if q5_words else 0
    q5_top5 = q5_word_counts.most_common(5)

    print_result("Q5 Word Count", dict(q5_word_counts))
    print_result("Q5 Character Count", dict(q5_char_counts))
    print_result("Q5 Average Word Length Per Word", q5_per_word_avg)
    print_result("Q5 Global Average Word Length", {"global_avg": q5_global_avg})
    print("\nQ5 Top 5 Most Frequent Words")
    for word, freq in q5_top5:
        print(f"{word}: {freq}")
else:
    print(f"File not found: {q5_file_path}")
    print("Download the file from the assignment link and update q5_file_path.")

if MRJOB_AVAILABLE:
    class MRTop5Words(MRJob):
        def mapper(self, _, line):
            for word in tokenize(line):
                yield word, 1

        def reducer(self, word, counts):
            yield None, (sum(counts), word)

        def reducer_top5(self, _, count_word_pairs):
            top5 = sorted(count_word_pairs, reverse=True)[:5]
            for count, word in top5:
                yield word, count

        def steps(self):
            return [
                MRStep(mapper=self.mapper, reducer=self.reducer),
                MRStep(reducer=self.reducer_top5)
            ]

    print("\nQ5 mrjob class ready: MRTop5Words")

File not found: sample_q5_input.txt
Download the file from the assignment link and update q5_file_path.

Q5 mrjob class ready: MRTop5Words


## Q6. Compute average marks for each student
Input:
- `A 80`
- `B 70`
- `A 90`
- `B 60`
- `A 100`

In [ ]:
q6_data = ["A 80", "B 70", "A 90", "B 60", "A 100"]

def q6_mapper(record):
    student, marks = record.split()
    return [(student, int(marks))]

def q6_reducer(student, marks):
    return sum(marks) / len(marks)

q6_manual = run_manual_mapreduce(q6_data, q6_mapper, q6_reducer)
print_result("Q6 Manual Output", q6_manual)

if MRJOB_AVAILABLE:
    class MRAverageMarks(MRJob):
        def mapper(self, _, line):
            student, marks = line.split()
            yield student, int(marks)

        def reducer(self, student, marks):
            marks = list(marks)
            yield student, sum(marks) / len(marks)

    print("\nQ6 mrjob class ready: MRAverageMarks")


Q6 Manual Output
A: 90.0
B: 65.0

Q6 mrjob class ready: MRAverageMarks


## Q7. Compute average salary per department and highest paid department
Input:
- `HR 50000`
- `IT 70000`
- `HR 60000`
- `IT 80000`

In [ ]:
q7_data = ["HR 50000", "IT 70000", "HR 60000", "IT 80000"]

def q7_mapper(record):
    dept, salary = record.split()
    return [(dept, int(salary))]

def q7_reducer(dept, salaries):
    return sum(salaries) / len(salaries)

q7_manual = run_manual_mapreduce(q7_data, q7_mapper, q7_reducer)
print_result("Q7 Average Salary Per Department", q7_manual)
highest_paid_dept = max(q7_manual.items(), key=lambda item: item[1])
print(f"\nHighest Paid Department: {highest_paid_dept[0]} ({highest_paid_dept[1]})")

if MRJOB_AVAILABLE:
    class MRDepartmentSalary(MRJob):
        def mapper(self, _, line):
            dept, salary = line.split()
            yield dept, int(salary)

        def reducer(self, dept, salaries):
            salaries = list(salaries)
            yield dept, sum(salaries) / len(salaries)

    print("\nQ7 mrjob class ready: MRDepartmentSalary")


Q7 Average Salary Per Department
HR: 55000.0
IT: 75000.0

Highest Paid Department: IT (75000.0)

Q7 mrjob class ready: MRDepartmentSalary


## Q8. Compute average temperature per country
Input:
- `New York,38`
- `London,29`
- `Tokyo,35`
- `New York,32`
- `Delhi,45`
- `Ambala,35`

The question says country, but the sample values are city names, so this solution groups by the first field provided in the input.

In [ ]:
q8_data = ["New York,38", "London,29", "Tokyo,35", "New York,32", "Delhi,45", "Ambala,35"]

def q8_mapper(record):
    place, temp = record.rsplit(',', 1)
    return [(place.strip(), float(temp))]

def q8_reducer(place, temps):
    return sum(temps) / len(temps)

q8_manual = run_manual_mapreduce(q8_data, q8_mapper, q8_reducer)
print_result("Q8 Manual Output", q8_manual)

if MRJOB_AVAILABLE:
    class MRTemperatureAverage(MRJob):
        def mapper(self, _, line):
            place, temp = line.rsplit(',', 1)
            yield place.strip(), float(temp)

        def reducer(self, place, temps):
            temps = list(temps)
            yield place, sum(temps) / len(temps)

    print("\nQ8 mrjob class ready: MRTemperatureAverage")


Q8 Manual Output
Ambala: 35.0
Delhi: 45.0
London: 29.0
New York: 35.0
Tokyo: 35.0

Q8 mrjob class ready: MRTemperatureAverage


## Q9. Redo Q8 using the Kaggle Global Land Temperatures dataset

Download the dataset locally and update `q9_csv_path`. This example uses a CSV with city and average temperature columns. Adjust the column names if your file differs.

In [ ]:
q9_csv_path = Path("GlobalLandTemperaturesByCity.csv")

def guess_country_from_row(row):
    for key in ["Country", "country"]:
        if key in row and row[key]:
            return row[key].strip()
    for key in ["City", "city"]:
        if key in row and row[key]:
            return row[key].strip()
    return None

def guess_temp_from_row(row):
    for key in ["AverageTemperature", "average_temperature", "Temp", "temperature"]:
        if key in row and row[key] not in (None, ""):
            try:
                return float(row[key])
            except ValueError:
                return None
    return None

if q9_csv_path.exists():
    q9_pairs = []
    with q9_csv_path.open("r", encoding="utf-8", errors="ignore") as f:
        reader = csv.DictReader(f)
        for row in reader:
            place = guess_country_from_row(row)
            temp = guess_temp_from_row(row)
            if place is not None and temp is not None:
                q9_pairs.append((place, temp))

    grouped = defaultdict(list)
    for place, temp in q9_pairs:
        grouped[place].append(temp)
    q9_manual = {place: sum(temps) / len(temps) for place, temps in grouped.items()}
    print("Q9 Manual Output (first 20 rows)")
    for i, (place, avg_temp) in enumerate(sorted(q9_manual.items())[:20], start=1):
        print(f"{i}. {place}: {avg_temp:.2f}")
else:
    print(f"File not found: {q9_csv_path}")
    print("Download the Kaggle CSV and update q9_csv_path.")

if MRJOB_AVAILABLE:
    class MRKaggleTemperatureAverage(MRJob):
        def mapper(self, _, line):
            if line.lower().startswith("dt,"):
                return
            parts = next(csv.reader([line]))
            if len(parts) < 4:
                return
            try:
                temp = float(parts[1])
            except ValueError:
                return
            country = parts[3].strip() if len(parts) > 3 else parts[2].strip()
            yield country, temp

        def reducer(self, country, temps):
            temps = list(temps)
            yield country, sum(temps) / len(temps)

    print("\nQ9 mrjob class ready: MRKaggleTemperatureAverage")

File not found: GlobalLandTemperaturesByCity.csv
Download the Kaggle CSV and update q9_csv_path.

Q9 mrjob class ready: MRKaggleTemperatureAverage
